# Lecture 3 — Regularization Strategies & Cross-Validation

**Statistisches Lernen 2** — FH Kufstein Tirol  
Professor: Johannes Schwab, PhD

---

## O que estamos aprendendo nesta aula (Resumo em 5 linhas)

Esta aula une o rigor estatístico às técnicas mais avançadas do Deep Learning moderno. Você entenderá que **regularizar um modelo** significa aplicar um filtro de sobriedade matemática aos seus parâmetros, seja restringindo-os geometricamente através de **círculos** (L2) ou **diamantes** (L1), ou sabotando o modelo intencionalmente para torná-lo robusto (**Dropout**). Veremos por que a restrição ideal de esparsidade (L0) é matematicamente intratável e como a L1 é sua **convexificação elegante**. Além disso, você aprenderá a validar essas escolhas usando o ecossistema de **Cross-Validation**, garantindo que as métricas de performance reflitam a realidade de produção e não apenas uma ilusão estatística.

---

## Roteiro do Notebook

1. **The Concept of Regularization** — Hard, Soft/Variational e Output Regularization  
2. **L2 Regularization (Ridge / Weight Decay)** — Geometria do círculo, Smoothness, solução analítica  
3. **The Sparsity Problem & L1 Regularization (Lasso)** — L0 é NP-hard; L1 como convexificação  
4. **Elastic Net** — Combinando L1 e L2  
5. **Modern Regularization Techniques** — Early Stopping, Dropout, Label Smoothing  
6. **The Probabilistic Interpretation (Bayesian Connection)** — Gaussian Prior → L2; Laplace Prior → L1  
7. **Rigorous Model Evaluation via Cross-Validation** — Hold-Out, K-Fold, Stratified, Leave-One-Out  
8. **Hyperparameter Tuning & Diagnosis Workflow** — Log-Grid, Under vs. Over-Regularization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LogNorm
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (
    KFold, StratifiedKFold, LeaveOneOut,
    cross_val_score, train_test_split, GridSearchCV
)
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.dummy import DummyRegressor
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('Bibliotecas carregadas com sucesso.')

---

## 1. The Concept of Regularization

### Conceito Teórico

**Regularization** (Regularização) é o conjunto de métodos para **controlar a complexidade do modelo** com o objetivo de melhorar a **generalização** — ou seja, o desempenho em dados novos, não vistos durante o treino.

**Analogia do estudante superinteligente:** imagine um aluno que é tão bom em memorizar que decora todas as respostas da lista de exercícios, incluindo os erros de digitação. Na prova, com questões ligeiramente diferentes, ele falha. Regularização é como um professor que diz: "você só pode usar fórmulas simples" — forçando o aluno a entender, não decorar.

O professor Schwab nos apresenta o **framework unificado** da Loss Function:

$$\mathcal{L}(\theta; X, Y) = \frac{1}{N} \sum_{i=1}^{N} \mathcal{D}(f_\theta(x_i), y_i)$$

Onde $\mathcal{D}(f_\theta(x_i), y_i) = \|f_\theta(x_i) - y_i\|^2$ é a função de distância (MSE).

Sem regularização, o otimizador encontra o $\theta^*$ que minimiza $\mathcal{L}$ nos dados de treino — potencialmente com **Overfitting** severo.

---

### 1a. Hard Regularization (Restrição Estrita de Conjunto)

**Conceito:** o espaço de parâmetros é **restrito a um conjunto** $C \subset \mathbb{R}^d$. O otimizador só pode explorar soluções dentro desse conjunto.

**Matemática:**

$$\min_{\theta} \mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 \quad \text{sujeito a} \quad \theta \in C \subset \mathbb{R}^d$$

A condição pode ser uma **igualdade** ($\theta_j = 0$ para Feature Selection) ou **desigualdade** ($\|\theta\| \leq R$).

**Why & How:** Hard Regularization é elegante matematicamente mas difícil de implementar com gradiente descendente padrão. É resolvida com Projected Gradient Descent ou Proximal Methods (tema da aula de Optimization).

---

### 1b. Soft / Variational Regularization (Termo de Penalidade)

**Conceito:** em vez de proibir regiões do espaço de parâmetros, **adicionamos uma penalidade** $\lambda \mathcal{R}(\theta)$ à Loss Function. O otimizador ainda pode ir a qualquer lugar, mas paga um custo crescente por parâmetros grandes.

**Matemática:**

$$\min_{\theta} \mathcal{L}(\theta) = \underbrace{\frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2}_{\text{Data Fit}} + \underbrace{\lambda \mathcal{R}(\theta)}_{\text{Penalty}}$$

Onde $\mathcal{R}: \mathbb{R}^d \to [0, \infty)$ é o funcional de regularização e $\lambda \geq 0$ é o **hiperparâmetro de regularização**:
- $\lambda = 0$: sem regularização, problema original
- $\lambda \to \infty$: apenas regularização, $\theta \to 0$

---

### 1c. Output Regularization

**Conceito:** em vez de penalizar os parâmetros $\theta$ diretamente, penalizamos propriedades da **saída** $f_\theta(x_i)$.

**Matemática:**

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \mathcal{R}(y_i)$$

**Exemplo prático:** em **Label Smoothing** (Seção 5), penalizamos saídas com confiança excessiva (probabilidades muito próximas de 0 ou 1).

In [ ]:
# Visualizando os três tipos de regularização como diagrama
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

theta_1 = np.linspace(-3, 3, 400)
theta_2 = np.linspace(-3, 3, 400)
T1, T2 = np.meshgrid(theta_1, theta_2)

# Loss function: elipses centradas em (2, 1.5) — ponto ótimo não-regularizado
Loss = (T1 - 2)**2 + 0.5*(T2 - 1.5)**2

# --- Hard Regularization: círculo L2 ---
ax = axes[0]
ax.contour(T1, T2, Loss, levels=15, cmap='RdYlGn_r', alpha=0.7)
circulo = plt.Circle((0, 0), 1.5, fill=False, color='navy', lw=2.5, ls='--')
ax.add_patch(circulo)
ax.fill(1.5*np.cos(np.linspace(0, 2*np.pi, 300)),
        1.5*np.sin(np.linspace(0, 2*np.pi, 300)),
        alpha=0.12, color='navy')
ax.scatter([2], [1.5], s=120, color='red', zorder=6, label='$\\theta^*$ (ótimo s/ reg.)')
ax.scatter([1.04], [1.08], s=120, color='navy', marker='*', zorder=6, label='$\\theta_{reg}$ (com restr.)')
ax.set_title('Hard Regularization\n$\\theta \\in B_R = \\{\\theta: \\|\\theta\\| \\leq R\\}$')
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.legend(fontsize=8); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')

# --- Soft Regularization: contorno da penalidade ---
ax = axes[1]
Penalty = T1**2 + T2**2  # L2 penalty
Total = Loss + 0.8 * Penalty
ax.contour(T1, T2, Loss, levels=12, cmap='RdYlGn_r', alpha=0.5, linestyles='--')
ax.contour(T1, T2, Total, levels=12, cmap='Blues', alpha=0.8)
ax.scatter([2], [1.5], s=120, color='red', zorder=6, label='$\\theta^*$ (sem $\\lambda$)')
ax.scatter([0.8], [0.65], s=120, color='blue', marker='*', zorder=6, label='$\\theta_{reg}$ (com $\\lambda$)')
ax.set_title('Soft / Variational Regularization\n$\\mathcal{L}(\\theta) + \\lambda\\mathcal{R}(\\theta)$')
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.legend(fontsize=8); ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)

# --- Output Regularization: penaliza o output ---
ax = axes[2]
prob = np.linspace(0.001, 0.999, 400)
# Cross-entropy loss para label = 1 (sem smoothing)
ce_hard   = -np.log(prob)
# Com Label Smoothing (epsilon = 0.1): target muda de 1.0 para 0.9
ce_smooth = -0.9*np.log(prob) - 0.1*np.log(1 - prob)
ax.plot(prob, ce_hard,   'tab:red',  lw=2.5, label='Hard label (target=1.0)')
ax.plot(prob, ce_smooth, 'tab:blue', lw=2.5, label='Label Smoothing (target=0.9)')
ax.set_xlabel('Probabilidade prevista $p$')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Output Regularization\nExemplo: Label Smoothing')
ax.set_ylim(0, 5); ax.legend(fontsize=9)

plt.suptitle('Os Três Tipos de Regularization — Visão Geral', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 2. L2 Regularization (Ridge / Weight Decay)

### Conceito Teórico

A **L2 Regularization** restringe a **norma euclidiana quadrática** dos parâmetros. Ela tem muitos nomes na literatura:
- **Ridge Regression** (em estatística)
- **Weight Decay** (em Deep Learning)
- **Tikhonov Regularization** (em matemática aplicada)

**Analogia da mola:** pense em cada coeficiente $\theta_j$ como um peso preso a uma mola ancorada na origem. Os dados puxam o peso para o valor ótimo, mas a mola resiste — não deixa o coeficiente crescer além do necessário.

### Matemática

**Hard form** (restrição de conjunto):

$$\min_\theta \mathcal{L}(\theta) \quad \text{s.t.} \quad \theta \in B_R = \{\theta : \|\theta\| \leq R\}$$

**Soft form** (variacional — a forma mais usada na prática):

$$\mathcal{L}(\theta) = \underbrace{\frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2}_{\text{MSE}} + \lambda\|\theta\|^2$$

**Extensão com transformação:** o funcional pode penalizar uma transformação dos parâmetros:

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \lambda\|T(\theta)\|^2$$

**Solução analítica** para regressão linear:

$$\hat{\theta}_{\text{Ridge}} = (X^TX + \lambda I)^{-1}X^Ty$$

Comparando com OLS: $(X^TX)^{-1}X^Ty$ — a regularização adiciona $\lambda I$ ao diagonal, garantindo que a matriz seja invertível mesmo quando $X^TX$ é singular.

### Efeito Geométrico — O Círculo

A região viável da Hard Regularization L2 é uma **bola euclidiana** (círculo em 2D). O otimizador encontra o ponto dentro dessa bola mais próximo do ótimo não-restrito $\theta^*$.

**Efeito de Smoothness:** L2 penaliza todos os coeficientes proporcionalmente à sua magnitude, **encolhendo** todos em direção a zero, mas **raramente zerando** algum exatamente.

### Why & How

- $\lambda = 0$: sem regularização, solução OLS
- $\lambda \to \infty$: $\hat{\theta} \to 0$ (underfitting total)
- Escolha de $\lambda$: sempre via **Cross-Validation** em escala logarítmica (Log-Grid)

In [ ]:
# Visualização geométrica: Hard L2 vs. Loss contours
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

T1, T2 = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-3, 3, 300))
# Loss centrada em (2, 1) — ponto ótimo não-regularizado
Loss = (T1 - 2.0)**2 + 0.4*(T2 - 1.0)**2

ax = axes[0]
cs = ax.contour(T1, T2, Loss, levels=np.linspace(0.1, 10, 20), cmap='RdYlGn_r', alpha=0.8)
theta_restrito = (2.0 * 1.5 / np.sqrt(2.0**2 + 1.0**2), 1.0 * 1.5 / np.sqrt(2.0**2 + 1.0**2))

for R, alpha, cor in [(2.5, 0.06, 'navy'), (1.5, 0.10, 'navy'), (0.8, 0.15, 'navy')]:
    theta_plot = np.linspace(0, 2*np.pi, 300)
    ax.fill(R*np.cos(theta_plot), R*np.sin(theta_plot), alpha=alpha, color=cor)
    ax.plot(R*np.cos(theta_plot), R*np.sin(theta_plot), color='navy', lw=1.5, alpha=0.7)

ax.scatter([2.0], [1.0], s=150, color='red', zorder=7, label='$\\theta^*$ (sem regularização)')
ax.scatter([theta_restrito[0]], [theta_restrito[1]], s=150, color='navy', marker='*', zorder=7,
           label='$\\theta_{L2}$ (com restrição $R=1.5$)')
ax.annotate('', xy=theta_restrito, xytext=(2.0, 1.0),
            arrowprops=dict(arrowstyle='->', color='purple', lw=2))
ax.text(0.5*(2.0+theta_restrito[0])+0.1, 0.5*(1.0+theta_restrito[1])+0.2,
        'Projeção\n(encolhimento)', color='purple', fontsize=9)
ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
ax.set_title('L2 Geometria: círculos concêntricos\n(Hard Regularization)')
ax.legend(fontsize=9); ax.set_aspect('equal')

# Direita: efeito de Ridge no encolhimento de coeficientes
ax = axes[1]
X_r, y_r = make_regression(n_samples=100, n_features=20, noise=10.0,
                            n_informative=5, random_state=42)
lambdas = np.logspace(-3, 4, 60)
coefs = []
for lam in lambdas:
    ridge = Ridge(alpha=lam)
    ridge.fit(X_r, y_r)
    coefs.append(ridge.coef_)
coefs = np.array(coefs)

for j in range(20):
    ax.plot(lambdas, coefs[:, j], lw=1.2, alpha=0.7)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set_xscale('log')
ax.set_xlabel('$\\lambda$ (força da regularização)')
ax.set_ylabel('Valor do coeficiente $\\theta_j$')
ax.set_title('Regularization Path — Ridge (L2)\nTodos os coeficientes encolhem suavemente')

plt.suptitle('L2 Regularization (Ridge / Weight Decay)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstração prática: Ridge em dados com alta dimensionalidade (p > N)
np.random.seed(7)
N, p = 50, 100  # mais features do que amostras
X_hd = np.random.randn(N, p)
theta_real = np.zeros(p)
theta_real[:5] = [3.0, -2.0, 1.5, -1.0, 0.8]  # apenas 5 features importam
y_hd = X_hd @ theta_real + np.random.normal(0, 0.5, N)

X_tr_hd, X_te_hd, y_tr_hd, y_te_hd = train_test_split(X_hd, y_hd, test_size=0.3, random_state=42)

lambdas_test = np.logspace(-4, 3, 30)
mse_treino_l2, mse_teste_l2 = [], []

for lam in lambdas_test:
    ridge = Ridge(alpha=lam)
    ridge.fit(X_tr_hd, y_tr_hd)
    mse_treino_l2.append(mean_squared_error(y_tr_hd, ridge.predict(X_tr_hd)))
    mse_teste_l2.append(mean_squared_error(y_te_hd, ridge.predict(X_te_hd)))

melhor_lam_idx = np.argmin(mse_teste_l2)

plt.figure(figsize=(10, 5))
plt.semilogx(lambdas_test, mse_treino_l2, 'tab:green', lw=2.5, marker='o', ms=4, label='MSE Treino')
plt.semilogx(lambdas_test, mse_teste_l2,  'tab:red',   lw=2.5, marker='s', ms=4, label='MSE Teste')
plt.axvline(lambdas_test[melhor_lam_idx], color='purple', ls='--', lw=2,
            label=f'$\\lambda^* = {lambdas_test[melhor_lam_idx]:.4f}$')
plt.xlabel('$\\lambda$ (Log-Scale)')
plt.ylabel('MSE')
plt.title(f'Ridge: efeito de $\\lambda$ em dados com p={p} > N={N}\n'
          f'Under-Regularization (esq.) vs. Over-Regularization (dir.)')
plt.legend(fontsize=10)
plt.tight_layout()
plt.show()
print(f"Melhor λ (no teste): {lambdas_test[melhor_lam_idx]:.4f}  →  MSE teste = {mse_teste_l2[melhor_lam_idx]:.4f}")
print(f"Sem regularização (λ→0): MSE teste = {mse_teste_l2[0]:.4f}")

---

## 3. The Sparsity Problem & L1 Regularization (Lasso)

### Conceito Teórico — Por que queremos Sparsity?

Muitas vezes, de centenas de features disponíveis, apenas algumas dezenas realmente importam para prever o target. Um modelo **esparso** (com muitos coeficientes exatamente zero) é:
- Mais interpretável ("quais features realmente importam?")
- Mais rápido (menos operações em inferência)
- Mais robusto (menos ruído acumulado de features irrelevantes)

**Analogia da mochila:** você vai acampar e quer levar apenas o essencial. L0 pergunta: "quais itens posso eliminar completamente?" (força de esparsidade real). L2 pergunta: "posso usar versões menores de todos os itens?" (encolhimento sem eliminação).

### Matemática — L0: A Restrição Ideal (NP-Hard)

A restrição perfeita de esparsidade seria a **L0 "norm"** (não é uma norma verdadeira — conta apenas o número de elementos não-zero):

$$B_R^0 = \{\theta : \|\theta\|_0 \leq R\} = \{\theta : \#\{\theta_i \neq 0\} \leq s\}$$

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \lambda\|\theta\|_0$$

**Por que L0 é inutilizável na prática?**
1. $\|\cdot\|_0$ **não é diferenciável** — gradiente não existe nos pontos de transição
2. $\|\cdot\|_0$ **não é convexa** — a região viável $B_R^0$ são as uniões dos eixos coordenados (ver slide "Sparsity geometrisch")
3. Otimização exata requer **enumerar todos os $\binom{p}{s}$ subconjuntos** de features → **NP-hard** (exponencial em $p$)

### Matemática — L1: A Convexificação Elegante

A **L1 Regularization** (Lasso) substitui a pseudo-norma L0 pela **norma L1** — o menor relaxamento convexo que ainda induz esparsidade:

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \lambda\|\theta\|_1 = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \lambda\sum_{j=1}^{d}|\theta_j|$$

**A região viável da Hard L1** é um **diamante (octaedro em d dimensões)**:

$$B_R^1 = \{\theta : \|\theta\|_1 \leq R\} = \{\theta : \sum_j |\theta_j| \leq R\}$$

**Por que o diamante produz esparsidade?** Os **vértices** do diamante estão exatamente nos eixos coordenados (pontos onde $\theta_j = 0$ para $j \neq k$). As curvas de nível da Loss Function tendem a tocar o diamante nesses vértices — forçando coeficientes a zero!

### Elastic Net — O Melhor dos Dois Mundos

**Elastic Net** combina L1 e L2:

$$\mathcal{L}(\theta) = \frac{1}{N}\sum_{i=1}^{N}\|f_\theta(x_i) - y_i\|^2 + \lambda_1\|\theta\|_1 + \lambda_2\|\theta\|^2$$

**Why & How:** quando features são correlacionadas, Lasso seleciona arbitrariamente uma delas. Elastic Net seleciona grupos de features correlacionadas juntas, mantendo a esparsidade.

In [ ]:
# Visualização geométrica: L2 (círculo) vs. L1 (diamante) vs. L0 (eixos)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

T1, T2 = np.meshgrid(np.linspace(-2.5, 2.5, 300), np.linspace(-2.5, 2.5, 300))
Loss = (T1 - 2.0)**2 + 0.5*(T2 - 1.5)**2

titulos = ['L0 (Sparsity — NP-hard)', 'L1 — Lasso (Diamante)', 'L2 — Ridge (Círculo)']
cores = ['tab:gray', 'tab:orange', 'navy']

for ax, titulo, cor in zip(axes, titulos, cores):
    ax.contour(T1, T2, Loss, levels=15, cmap='RdYlGn_r', alpha=0.7)
    ax.scatter([2.0], [1.5], s=120, color='red', zorder=7, label='$\\theta^*$ ótimo')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_xlabel('$\\theta_1$'); ax.set_ylabel('$\\theta_2$')
    ax.set_aspect('equal')
    ax.set_title(titulo)

# L0: eixos coordenados (union dos eixos para sparsity level 1)
ax = axes[0]
ax.axhline(0, color='gray', lw=4, alpha=0.5, label='$B_R^0$: eixos (s=1)')
ax.axvline(0, color='gray', lw=4, alpha=0.5)
ax.scatter([1.5], [0], s=120, color='gray', marker='*', zorder=7, label='$\\theta_{L0}$ (no eixo)')
ax.annotate('NP-Hard:\nnão é convexa\nnão é diferenciável',
            xy=(0.2, 0.8), fontsize=9, color='darkred', fontweight='bold')
ax.legend(fontsize=8)

# L1: diamante
ax = axes[1]
R1 = 1.5
diamond_x = [R1, 0, -R1, 0, R1]
diamond_y = [0, R1,  0, -R1, 0]
ax.fill(diamond_x, diamond_y, alpha=0.15, color='tab:orange')
ax.plot(diamond_x, diamond_y, color='tab:orange', lw=2.5)
# O ponto de toque está no vértice (θ1=0) → sparsidade!
ax.scatter([0], [R1], s=160, color='tab:orange', marker='*', zorder=7,
           label='$\\theta_{L1}$: toca o vértice!\n(esparsidade: $\\theta_1=0$)')
ax.annotate('Vértices nos\neixos → esparsidade!',
            xy=(0.05, R1-0.1), xytext=(0.5, 1.8),
            arrowprops=dict(arrowstyle='->', color='tab:orange'), color='tab:orange', fontsize=9)
ax.legend(fontsize=8)

# L2: círculo
ax = axes[2]
theta_c = np.linspace(0, 2*np.pi, 300)
ax.fill(R1*np.cos(theta_c), R1*np.sin(theta_c), alpha=0.12, color='navy')
ax.plot(R1*np.cos(theta_c), R1*np.sin(theta_c), color='navy', lw=2.5)
theta_l2 = np.array([2.0, 1.5]) * R1 / np.sqrt(2.0**2 + 1.5**2)
ax.scatter(theta_l2[0], theta_l2[1], s=160, color='navy', marker='*', zorder=7,
           label='$\\theta_{L2}$: toca a borda\n(sem esparsidade exata)')
ax.legend(fontsize=8)

plt.suptitle('Geometria das Restrições: L0 vs. L1 vs. L2', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Regularization Paths: Ridge vs. Lasso vs. Elastic Net
X_sp, y_sp = make_regression(n_samples=150, n_features=20, noise=8.0,
                              n_informative=4, random_state=42)

lambdas_path = np.logspace(-4, 2, 80)

coefs_ridge = [Ridge(alpha=lam).fit(X_sp, y_sp).coef_ for lam in lambdas_path]
coefs_lasso = [Lasso(alpha=lam, max_iter=10000).fit(X_sp, y_sp).coef_ for lam in lambdas_path]
coefs_enet  = [ElasticNet(alpha=lam, l1_ratio=0.5, max_iter=10000).fit(X_sp, y_sp).coef_
               for lam in lambdas_path]
coefs_ridge = np.array(coefs_ridge)
coefs_lasso = np.array(coefs_lasso)
coefs_enet  = np.array(coefs_enet)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
configs = [
    (axes[0], coefs_ridge, 'Ridge (L2)\nEncolhimento suave — nunca zera exatamente', 'tab:blue'),
    (axes[1], coefs_lasso, 'Lasso (L1)\nEsparsidade: coeficientes vão a zero!', 'tab:orange'),
    (axes[2], coefs_enet,  'Elastic Net (L1 + L2)\nEsparsidade + grupos de features', 'tab:green'),
]

for ax, coefs, titulo, cor in configs:
    for j in range(20):
        alpha_j = 0.9 if np.max(np.abs(coefs[:, j])) > 5 else 0.4
        ax.plot(lambdas_path, coefs[:, j], lw=1.5, alpha=alpha_j)
    ax.axhline(0, color='black', lw=1.5, ls='--')
    ax.set_xscale('log')
    ax.set_xlabel('$\\lambda$ (Log-Scale)')
    ax.set_ylabel('$\\theta_j$')
    ax.set_title(titulo)
    ax.invert_xaxis()

plt.suptitle('Regularization Paths: Ridge vs. Lasso vs. Elastic Net\n'
             '(lendo da direita para esquerda: aumentando λ → mais regularização)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
# Contar coeficientes zero do Lasso no λ máximo
n_zeros = np.sum(np.abs(coefs_lasso[-1]) < 1e-6)
print(f"Lasso com λ={lambdas_path[-1]:.2f}: {n_zeros} de {20} coeficientes zerados (esparsidade = {n_zeros/20*100:.0f}%)")

---

## 4. Modern Regularization Techniques

### 4a. Early Stopping

### Conceito Teórico

**Early Stopping** reduz a Variance interrompendo o processo de otimização **antes** que o otimizador encontre o minimizador $\theta^*$ da Loss Function de treino.

**Analogia:** é como estudar para uma prova. Se você para de estudar na hora certa (quando entende o conceito), vai bem. Se continua estudando até decorar cada detalhe (incluindo erros do livro), começa a piorar porque está memorizando o irrelevante.

**Como funciona:** durante o treinamento iterativo (Gradient Descent):
1. Avalie o erro de **validação** a cada época
2. Se o erro de validação não melhora por `patience` épocas consecutivas → **pare**
3. Restaure os pesos do **melhor ponto** (não do último)

**Matemática — Equivalência com L2:**

Para modelos lineares otimizados com Gradient Descent com taxa de aprendizado $\eta$ e $t$ iterações:

$$\hat{\theta}_{\text{Early Stop}}(t) \approx \hat{\theta}_{\text{Ridge}}\left(\lambda = \frac{1}{\eta t}\right)$$

Parar cedo é **matematicamente equivalente à L2 Regularization** — mas sem o custo de escolher $\lambda$ explicitamente!

In [ ]:
# Simulando Early Stopping com Gradient Descent em regressão polinomial
f_true = lambda x: np.sin(2 * np.pi * x)

np.random.seed(13)
x_tr = np.sort(np.random.uniform(0, 1, 30))
y_tr = f_true(x_tr) + np.random.normal(0, 0.2, 30)
x_val = np.sort(np.random.uniform(0, 1, 20))
y_val = f_true(x_val) + np.random.normal(0, 0.2, 20)

# Design matrix polinomial grau 15
grau = 15
def poly_features(x, g):
    return np.column_stack([x**k for k in range(g+1)])

B_tr  = poly_features(x_tr, grau)
B_val = poly_features(x_val, grau)

# Gradient Descent manual
theta = np.zeros(grau + 1)
eta = 1e-3
n_epocas = 5000
patience = 200

historico_treino, historico_val = [], []
melhor_val, melhor_theta, melhor_epoca = np.inf, theta.copy(), 0
sem_melhora = 0
epoca_stop = n_epocas

for ep in range(n_epocas):
    grad = -2/len(x_tr) * B_tr.T @ (y_tr - B_tr @ theta)
    theta = theta - eta * grad
    mse_t = mean_squared_error(y_tr, B_tr @ theta)
    mse_v = mean_squared_error(y_val, B_val @ theta)
    historico_treino.append(mse_t)
    historico_val.append(mse_v)
    if mse_v < melhor_val:
        melhor_val = mse_v
        melhor_theta = theta.copy()
        melhor_epoca = ep
        sem_melhora = 0
    else:
        sem_melhora += 1
        if sem_melhora >= patience and epoca_stop == n_epocas:
            epoca_stop = ep

x_plot = np.linspace(0, 1, 300)
B_plot = poly_features(x_plot, grau)
theta_final = theta  # último ponto (overfitting)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogy(historico_treino, 'tab:green', lw=2, label='Loss Treino')
ax.semilogy(historico_val,   'tab:red',   lw=2, label='Loss Validação')
ax.axvline(melhor_epoca, color='blue', ls='--', lw=2, label=f'Early Stopping (época {melhor_epoca})')
ax.axvline(epoca_stop, color='purple', ls=':', lw=2, label=f'Patience esgotada (época {epoca_stop})')
ax.set_xlabel('Época'); ax.set_ylabel('MSE (log)')
ax.set_title('Early Stopping:\nMonitorando Loss de Validação')
ax.legend(fontsize=9)

ax = axes[1]
ax.scatter(x_tr, y_tr, s=40, color='gray', alpha=0.7, zorder=5, label='Treino')
ax.plot(x_plot, f_true(x_plot), 'k--', alpha=0.4, lw=1.5, label='$f$ verdadeira')
ax.plot(x_plot, B_plot @ theta_final, 'tab:red', lw=2, alpha=0.8, label=f'Sem Early Stop (época {n_epocas})')
ax.plot(x_plot, B_plot @ melhor_theta, 'tab:blue', lw=2.5, label=f'Com Early Stop (época {melhor_epoca})')
ax.set_ylim(-2.5, 2.5); ax.set_title('Curva ajustada: com vs. sem Early Stopping')
ax.legend(fontsize=9)

plt.suptitle('Early Stopping — Regularização via Interrupção da Otimização', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 4b. Dropout

### Conceito Teórico

**Dropout** é uma técnica de regularização para redes neurais: durante cada iteração de treino, zeramos aleatoriamente uma fração $p$ dos neurônios (suas ativações são forçadas a zero).

**Analogia do time de futebol:** imagine treinar cada jogador para jogar bem mesmo sem alguns companheiros de time. Quando o time completo jogar, cada jogador será mais resiliente e versátil — não depende de parceiros específicos.

**Por que funciona como Ensemble?** Com $d$ neurônios e taxa $p$, existem $2^d$ sub-redes possíveis. O Dropout treina implicitamente uma **média exponencial de modelos** — equivalente a um Ensemble sem o custo de treinar cada modelo separadamente.

### Matemática

Para uma camada com ativação $\mathbf{h}$, o Dropout aplica uma máscara binária:

$$\tilde{h}_j = m_j \cdot h_j, \quad m_j \sim \text{Bernoulli}(1-p)$$

Durante **inferência**, não há dropout — mas os pesos são escalados por $(1-p)$ para compensar a média:

$$h_j^{\text{test}} = (1-p) \cdot h_j$$

**Why & How:** Dropout é padrão em redes neurais profundas. Taxa típica: $p = 0.5$ para camadas ocultas densas, $p = 0.1$-$0.2$ para camadas convolucionais.

In [ ]:
# Simulação de Dropout: efeito no vetor de ativações
np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

h = np.random.randn(20)  # ativações de 20 neurônios

for ax, p_drop, titulo in [
    (axes[0], 0.0, 'Sem Dropout (p=0)\nTreino normal'),
    (axes[1], 0.5, 'Dropout p=0.5\n~50% dos neurônios zerados'),
    (axes[2], 0.8, 'Dropout p=0.8\n~80% dos neurônios zerados'),
]:
    mask = np.random.binomial(1, 1 - p_drop, size=len(h))
    h_dropped = h * mask
    # Normalização de inferência
    h_inference = h * (1 - p_drop)

    cores_bar = ['tab:blue' if m == 1 else 'tab:red' for m in mask]
    ax.bar(range(len(h)), h_dropped, color=cores_bar, alpha=0.8)
    ax.axhline(0, color='black', lw=1)
    n_ativos = int(np.sum(mask))
    ax.set_title(f'{titulo}\n({n_ativos}/{len(h)} neurônios ativos)')
    ax.set_xlabel('Índice do neurônio')
    ax.set_ylabel('Ativação')

# Legenda manual
azul_patch = mpatches.Patch(color='tab:blue', label='Neurônio ativo')
verm_patch = mpatches.Patch(color='tab:red',  label='Neurônio zerado (dropped)')
axes[2].legend(handles=[azul_patch, verm_patch], fontsize=9, loc='upper right')

plt.suptitle('Dropout: Regularização por Sabotagem Aleatória durante o Treino', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Intuição do Ensemble:")
print(f"  Com d=20 neurônios, o Dropout explora até 2^20 = {2**20:,} sub-redes possíveis")
print("  → equivalente a um Ensemble implícito de modelos!")

### 4c. Label Smoothing

### Conceito Teórico

Em classificação, os labels são normalmente **One-Hot** (valores binários: 0 ou 1). Um modelo treinado assim tende a ficar **superconfiante** — prevendo probabilidades muito próximas de 1.0 para a classe correta, o que causa overfitting.

**Label Smoothing** substitui os targets rígidos por distribuições suaves:

$$y_k^{\text{smooth}} = y_k (1 - \epsilon) + \frac{\epsilon}{K}$$

Onde $\epsilon$ é o parâmetro de suavização e $K$ é o número de classes.

**Exemplo com 3 classes e $\epsilon = 0.1$:**
- Hard label: $[1, 0, 0]$
- Smooth label: $[0.933, 0.033, 0.033]$

**Why & How:** isso é uma forma de **Output Regularization** — penaliza o modelo por ser excessivamente confiante. Usado principalmente em transformers e classificação de imagens. Valor típico: $\epsilon \in [0.05, 0.2]$.

In [ ]:
# Demonstrando Label Smoothing
K = 5  # 5 classes
epsilons = [0.0, 0.05, 0.1, 0.2]

fig, axes = plt.subplots(1, len(epsilons), figsize=(16, 4))

y_hard = np.array([1, 0, 0, 0, 0])  # classe 0 correta

for ax, eps in zip(axes, epsilons):
    y_smooth = y_hard * (1 - eps) + eps / K
    cores = ['tab:green' if i == 0 else 'tab:gray' for i in range(K)]
    bars = ax.bar(range(K), y_smooth, color=cores, alpha=0.85, edgecolor='black')
    for bar, v in zip(bars, y_smooth):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')
    ax.set_ylim(0, 1.1)
    ax.set_title(f'$\\epsilon = {eps}$\n{"One-Hot (hard)" if eps == 0 else "Label Smoothing"}')
    ax.set_xlabel('Classe'); ax.set_xticks(range(K))
    ax.set_xticklabels([f'C{i}' for i in range(K)])

axes[0].set_ylabel('Probabilidade do Target')
plt.suptitle('Label Smoothing: de One-Hot rígido para Distribuição Suave (Output Regularization)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 5. The Probabilistic Interpretation — The Bayesian Connection

### Conceito Teórico

Aqui está um dos resultados mais elegantes do aprendizado de máquina: **toda regularização é equivalente a um prior Bayesiano sobre os parâmetros**. Em outras palavras, quando você escolhe L2, está secretamente dizendo "acredito que os pesos verdadeiros são pequenos e gaussianos".

**Analogia do detetive bayesiano:** antes de ver qualquer evidência (dados), o detetive tem uma suspeita inicial (prior). Depois de ver as evidências (likelihood), ele atualiza sua crença (posterior). A regularização é a suspeita inicial codificada matematicamente.

### Matemática — Derivação Formal

**Ponto de partida:** Teorema de Bayes

$$p(\theta \mid \mathcal{D}) = \frac{p(\mathcal{D} \mid \theta) \cdot p(\theta)}{p(\mathcal{D})}$$

Queremos o estimador **MAP (Maximum A Posteriori)**:

$$\hat{\theta}_{\text{MAP}} = \arg\max_\theta \left[\log p(\mathcal{D} \mid \theta) + \log p(\theta)\right]$$

Ou equivalentemente (minimizando o negativo):

$$\hat{\theta}_{\text{MAP}} = \arg\min_\theta \left[-\log p(\mathcal{D} \mid \theta) - \log p(\theta)\right]$$

**Termo de likelihood** (assumindo ruído Gaussiano):

$$-\log p(\mathcal{D} \mid \theta) = \frac{1}{2\sigma^2}\sum_{i=1}^N (y_i - f_\theta(x_i))^2 + \text{const}$$

Este é o **MSE** (até uma constante)!

---

### Prova 1: Gaussian Prior → L2 Regularization

Suponha um prior Gaussiano isotrópico sobre os pesos:

$$p(\theta) = \mathcal{N}(0, \tau^2 I) = \prod_{j=1}^d \frac{1}{\sqrt{2\pi\tau^2}} \exp\!\left(-\frac{\theta_j^2}{2\tau^2}\right)$$

O log-prior é:

$$-\log p(\theta) = \frac{1}{2\tau^2}\sum_{j=1}^d \theta_j^2 + \text{const} = \frac{1}{2\tau^2}\|\theta\|^2 + \text{const}$$

O objetivo MAP completo:

$$\hat{\theta}_{\text{MAP}} = \arg\min_\theta \underbrace{\frac{1}{2\sigma^2}\sum_{i=1}^N (y_i - f_\theta(x_i))^2}_{\text{Negative Log-Likelihood}} + \underbrace{\frac{1}{2\tau^2}\|\theta\|^2}_{\text{L2 Penalty com } \lambda = \sigma^2/\tau^2}$$

$$\boxed{\text{Gaussian Prior} \equiv \text{L2 Regularization com } \lambda = \frac{\sigma^2}{\tau^2}}$$

---

### Prova 2: Laplace Prior → L1 Regularization

Suponha um prior de Laplace:

$$p(\theta) = \prod_{j=1}^d \frac{1}{2b} \exp\!\left(-\frac{|\theta_j|}{b}\right)$$

O log-prior é:

$$-\log p(\theta) = \frac{1}{b}\sum_{j=1}^d |\theta_j| + \text{const} = \frac{1}{b}\|\theta\|_1 + \text{const}$$

O objetivo MAP completo:

$$\hat{\theta}_{\text{MAP}} = \arg\min_\theta \frac{1}{2\sigma^2}\sum_{i=1}^N (y_i - f_\theta(x_i))^2 + \frac{1}{b}\|\theta\|_1$$

$$\boxed{\text{Laplace Prior} \equiv \text{L1 Regularization (Lasso) com } \lambda = \frac{\sigma^2}{b}}$$

**Insight geométrico:** o Prior de Laplace tem um **pico pontiagudo em zero** — ele *acredita fortemente* que a maioria dos pesos deveria ser zero. Esta crença se manifesta como esparsidade na solução MAP.

In [ ]:
# Visualizando os priors e suas conexões com as penalidades
from scipy.stats import norm, laplace

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

theta_vis = np.linspace(-4, 4, 400)

# --- Gaussian Prior ---
ax = axes[0]
for tau, lw_val, alpha_val in [(0.5, 3, 1.0), (1.0, 2, 0.7), (2.0, 1.5, 0.5)]:
    ax.plot(theta_vis, norm.pdf(theta_vis, 0, tau), lw=lw_val, alpha=alpha_val,
            label=f'$\\mathcal{{N}}(0, {tau}^2)$')
ax.set_title('Gaussian Prior\n$p(\\theta) = \\mathcal{N}(0, \\tau^2 I)$\n→ L2 Regularization')
ax.set_xlabel('$\\theta$'); ax.set_ylabel('$p(\\theta)$')
ax.legend(fontsize=9)
ax.axvline(0, color='gray', ls=':', lw=1)

# --- Laplace Prior ---
ax = axes[1]
for b, lw_val, alpha_val in [(0.5, 3, 1.0), (1.0, 2, 0.7), (2.0, 1.5, 0.5)]:
    ax.plot(theta_vis, laplace.pdf(theta_vis, 0, b), lw=lw_val, alpha=alpha_val,
            label=f'Laplace(0, {b})')
ax.set_title('Laplace Prior\n$p(\\theta) \\propto e^{-|\\theta|/b}$\n→ L1 Regularization (Lasso)')
ax.set_xlabel('$\\theta$'); ax.set_ylabel('$p(\\theta)$')
ax.legend(fontsize=9)
ax.axvline(0, color='gray', ls=':', lw=1)

# --- Comparação das penalidades ---
ax = axes[2]
ax.plot(theta_vis, theta_vis**2, 'tab:blue', lw=2.5, label='$\\|\\theta\\|^2$ (L2 / Gaussian)')
ax.plot(theta_vis, np.abs(theta_vis), 'tab:orange', lw=2.5, label='$\\|\\theta\\|_1$ (L1 / Laplace)')
ax.plot(theta_vis, (np.abs(theta_vis) > 0).astype(float), 'tab:gray',
        lw=2, ls='--', label='$\\|\\theta\\|_0$ (L0 — NP-hard)')
ax.set_xlim(-3, 3); ax.set_ylim(-0.1, 4)
ax.set_title('Penalidades: $-\\log p(\\theta)$ como função de $\\theta$\nGeometria das priors')
ax.set_xlabel('$\\theta$'); ax.set_ylabel('$-\\log p(\\theta)$')
ax.legend(fontsize=9)
ax.annotate('Pico em zero:\nacredita fortemente\nque θ ≈ 0', xy=(0, 0), xytext=(0.8, 2),
            arrowprops=dict(arrowstyle='->', color='tab:orange'), color='tab:orange', fontsize=9)

plt.suptitle('The Bayesian Connection: Prior → Regularization\n'
             'Gaussian Prior ≡ L2  |  Laplace Prior ≡ L1', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---

## 6. Rigorous Model Evaluation via Cross-Validation

### Conceito Teórico

**Cross-Validation** é o conjunto de técnicas para **estimar o erro de generalização** de forma confiável, sem contaminar o conjunto de teste. O princípio fundamental: **nunca use o conjunto de teste para tomar decisões** — nem para escolher $\lambda$, nem para comparar modelos.

**Analogia do exame escolar:**
- **Dados de treino** = lista de exercícios (aprende)
- **Dados de validação** = simulado (ajusta a estratégia)
- **Dados de teste** = prova final (avalia uma única vez)

Cross-Validation simula o "simulado" de formas diferentes para maximizar o uso dos dados disponíveis.

---

### 6a. Hold-Out

**Como funciona:** divide os dados uma única vez em treino (ex: 80%) e validação (ex: 20%).

**Vantagem:** rápido e simples. **Desvantagem:** estimativa instável (depende do split aleatório).

**Quando usar:** quando $N$ é muito grande e o custo computacional do K-Fold é proibitivo.

### 6b. K-Fold Cross-Validation

**Como funciona:** divide os dados em $K$ partes (**folds**) iguais. Treina $K$ vezes — cada vez usa $K-1$ folds para treino e 1 fold para validação. O erro final é a **média** dos $K$ erros.

$$\text{CV}_K = \frac{1}{K}\sum_{k=1}^K \text{MSE}_k$$

**Trade-off de K:**
- K pequeno (ex: 3): rápido, mas estimativa mais variável
- K grande (ex: 10): estimativa mais confiável, mais lento
- **K = 5 ou K = 10** são os valores mais usados na prática

### 6c. Stratified K-Fold

**Como funciona:** como K-Fold, mas garante que **cada fold preserve a proporção original de classes** (para dados desbalanceados).

**Por que importa:** se 5% dos dados são classe positiva e você usa K-Fold padrão, alguns folds podem ter 0% ou 10% de positivos — estimativas instáveis. Stratified garante ~5% em cada fold.

### 6d. Leave-One-Out (LOO)

**Como funciona:** caso extremo de K-Fold com $K = N$. Cada fold usa $N-1$ amostras para treino e 1 para validação.

**Vantagem:** estimativa quase sem viés (usa quase todos os dados para treino). **Desvantagem:** custo computacional $O(N)$ vezes o custo de treino — impraticável para $N$ grande.

In [ ]:
# Visualizando os padrões de divisão de cada método de CV
N_vis = 30
K = 5

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

# Cores: verde = treino, vermelho = validação/teste
COR_TREINO = '#4CAF50'
COR_VAL    = '#F44336'
COR_TESTE  = '#2196F3'

# --- Hold-Out ---
ax = axes[0]
split = int(0.8 * N_vis)
cores_ho = [COR_TREINO]*split + [COR_VAL]*(N_vis - split)
ax.barh([0]*N_vis, [1]*N_vis, left=range(N_vis), color=cores_ho, edgecolor='white', height=0.6)
ax.set_yticks([0]); ax.set_yticklabels(['Hold-Out'])
ax.set_xlim(0, N_vis); ax.set_title(f'Hold-Out (80/20 split)')
ax.set_xlabel('Índice das amostras')
treino_p = mpatches.Patch(color=COR_TREINO, label='Treino')
val_p    = mpatches.Patch(color=COR_VAL,    label='Validação')
ax.legend(handles=[treino_p, val_p], loc='lower right', fontsize=9)

# --- K-Fold ---
ax = axes[1]
kf = KFold(n_splits=K, shuffle=False)
indices = np.arange(N_vis)
for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(indices)):
    cores_fold = np.array([COR_TREINO]*N_vis)
    cores_fold[val_idx] = COR_VAL
    ax.barh([fold_idx]*N_vis, [1]*N_vis, left=range(N_vis),
            color=cores_fold, edgecolor='white', height=0.6)
ax.set_yticks(range(K)); ax.set_yticklabels([f'Fold {k+1}' for k in range(K)])
ax.set_title(f'{K}-Fold Cross-Validation')
ax.set_xlabel('Índice das amostras'); ax.set_xlim(0, N_vis)

# --- Stratified K-Fold (com classes) ---
ax = axes[2]
# Dataset artificial desbalanceado: 20% positivos
y_strat = np.array([0]*24 + [1]*6)  # 80% classe 0, 20% classe 1
np.random.shuffle(y_strat)
skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)
for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(np.zeros(N_vis), y_strat)):
    cores_strat = np.array([COR_TREINO]*N_vis)
    cores_strat[val_idx] = COR_VAL
    ax.barh([fold_idx]*N_vis, [1]*N_vis, left=range(N_vis),
            color=cores_strat, edgecolor='white', height=0.6)
    n_pos_val = np.sum(y_strat[val_idx] == 1)
    ax.text(N_vis + 0.5, fold_idx, f'{n_pos_val}/{len(val_idx)} pos.',
            va='center', fontsize=8, color='purple')
ax.set_yticks(range(K)); ax.set_yticklabels([f'Fold {k+1}' for k in range(K)])
ax.set_title(f'Stratified {K}-Fold (preserva proporção de classes ~20% positivos)')
ax.set_xlabel('Índice das amostras'); ax.set_xlim(0, N_vis + 4)

# --- Leave-One-Out ---
ax = axes[3]
N_loo = 15  # menor para visualizar
for sample_idx in range(min(N_loo, 10)):  # mostrar só 10 primeiros
    cores_loo = np.array([COR_TREINO]*N_loo)
    cores_loo[sample_idx] = COR_VAL
    ax.barh([sample_idx]*N_loo, [1]*N_loo, left=range(N_loo),
            color=cores_loo, edgecolor='white', height=0.6)
ax.set_yticks(range(10)); ax.set_yticklabels([f'LOO-{k}' for k in range(10)])
ax.set_title(f'Leave-One-Out (LOO) — primeiros 10 de {N_loo} iterações (K=N={N_loo})')
ax.set_xlabel('Índice das amostras'); ax.set_xlim(0, N_loo)

plt.suptitle('Variantes de Cross-Validation — Padrões de Divisão', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Comparação quantitativa: Hold-Out vs. K-Fold vs. LOO
# Usando Ridge para comparar estimativas de erro
X_cv, y_cv = make_regression(n_samples=80, n_features=10, noise=15.0,
                              n_informative=5, random_state=42)

modelo_cv = Ridge(alpha=1.0)
n_repeticoes = 50
resultados_cv = {'Hold-Out': [], 'K-Fold (K=5)': [], 'K-Fold (K=10)': [], 'LOO': []}

for seed in range(n_repeticoes):
    np.random.seed(seed)
    # Hold-Out
    X_tr, X_te, y_tr, y_te = train_test_split(X_cv, y_cv, test_size=0.2, random_state=seed)
    modelo_cv.fit(X_tr, y_tr)
    resultados_cv['Hold-Out'].append(mean_squared_error(y_te, modelo_cv.predict(X_te)))

    # K-Fold 5
    s5 = cross_val_score(modelo_cv, X_cv, y_cv, cv=KFold(5, shuffle=True, random_state=seed),
                         scoring='neg_mean_squared_error')
    resultados_cv['K-Fold (K=5)'].append(-s5.mean())

    # K-Fold 10
    s10 = cross_val_score(modelo_cv, X_cv, y_cv, cv=KFold(10, shuffle=True, random_state=seed),
                          scoring='neg_mean_squared_error')
    resultados_cv['K-Fold (K=10)'].append(-s10.mean())

# LOO (uma vez, não repetir — é determinístico)
s_loo = cross_val_score(modelo_cv, X_cv, y_cv, cv=LeaveOneOut(),
                        scoring='neg_mean_squared_error')
mse_loo = -s_loo.mean()
resultados_cv['LOO'] = [mse_loo] * n_repeticoes  # constante

fig, ax = plt.subplots(figsize=(11, 5))
metodos = list(resultados_cv.keys())
dados_plot = [resultados_cv[m] for m in metodos]
bp = ax.boxplot(dados_plot, labels=metodos, patch_artist=True,
                medianprops=dict(color='black', lw=2))
cores_box = ['tab:red', 'tab:blue', 'tab:green', 'tab:purple']
for patch, cor in zip(bp['boxes'], cores_box):
    patch.set_facecolor(cor); patch.set_alpha(0.5)

ax.set_ylabel('MSE estimado')
ax.set_title(f'Comparação de CV ({n_repeticoes} repetições com seeds diferentes)\n'
             'Variabilidade menor = estimativa mais confiável')

# Adicionar desvio padrão como texto
for i, (m, dados) in enumerate(resultados_cv.items()):
    std = np.std(dados)
    med = np.median(dados)
    ax.text(i+1, ax.get_ylim()[1]*0.97, f'std={std:.1f}', ha='center', fontsize=9,
            color=cores_box[i], fontweight='bold')

plt.tight_layout()
plt.show()
print("Análise das variabilidades:")
for m, dados in resultados_cv.items():
    print(f"  {m:<20}: média={np.mean(dados):.2f}, std={np.std(dados):.2f}")

---

## 7. Hyperparameter Tuning & Diagnosis Workflow

### Conceito Teórico — Por que Log-Grid?

Ao sintonizar $\lambda$, o impacto da regularização opera em **ordens de magnitude**, não em escala linear. A diferença entre $\lambda = 0.001$ e $\lambda = 0.01$ é enorme (fator 10×), enquanto a diferença entre $\lambda = 0.1$ e $\lambda = 0.2$ é mínima.

**Log-Grid** cobre uniformemente o espaço logarítmico:

$$\lambda \in \{10^{-4}, 10^{-3}, 10^{-2}, 10^{-1}, 10^0, 10^1, 10^2\}$$

**Estratégia Coarse-to-Fine:**
1. **Coarse sweep:** Log-Grid amplo para encontrar a ordem de magnitude certa
2. **Fine sweep:** Log-Grid estreito ao redor do melhor valor encontrado

### Diagnóstico de Curvas de Aprendizado

| Padrão | Diagnóstico | Ação |
|--------|-------------|------|
| Erro treino baixo, erro val. alto, gap grande | **Under-Regularization** (High Variance) | Aumentar $\lambda$ |
| Erro treino alto, erro val. alto, gap pequeno | **Over-Regularization** (High Bias) | Diminuir $\lambda$ |
| Erro treino ≈ erro val. ≈ Irreducible Error | **Ótimo** | Manter $\lambda$ |

In [ ]:
# Coarse-to-Fine Log-Grid search para λ do Ridge
np.random.seed(42)
X_tune, y_tune = make_regression(n_samples=200, n_features=30, noise=12.0,
                                  n_informative=8, random_state=42)
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_tune, y_tune, test_size=0.25, random_state=42)

# --- Coarse sweep ---
lambdas_coarse = np.logspace(-5, 5, 20)
scores_coarse  = []
kf_tune = KFold(n_splits=5, shuffle=True, random_state=42)
for lam in lambdas_coarse:
    s = cross_val_score(Ridge(alpha=lam), X_tr_t, y_tr_t, cv=kf_tune,
                        scoring='neg_mean_squared_error')
    scores_coarse.append(-s.mean())

melhor_coarse = lambdas_coarse[np.argmin(scores_coarse)]

# --- Fine sweep ---
lambdas_fine = np.logspace(np.log10(melhor_coarse) - 1, np.log10(melhor_coarse) + 1, 30)
scores_fine  = []
for lam in lambdas_fine:
    s = cross_val_score(Ridge(alpha=lam), X_tr_t, y_tr_t, cv=kf_tune,
                        scoring='neg_mean_squared_error')
    scores_fine.append(-s.mean())

melhor_fine = lambdas_fine[np.argmin(scores_fine)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.semilogx(lambdas_coarse, scores_coarse, 'tab:blue', lw=2.5, marker='o', ms=7,
            label='CV MSE (Coarse)')
ax.axvline(melhor_coarse, color='red', ls='--', lw=2, label=f'Melhor coarse: $\\lambda={melhor_coarse:.4f}$')
ax.set_xlabel('$\\lambda$ (Log-Scale)'); ax.set_ylabel('MSE (CV)')
ax.set_title('Passo 1: Coarse Sweep\n(Log-Grid amplo: $10^{-5}$ a $10^5$)')
ax.legend(fontsize=9)

ax = axes[1]
ax.semilogx(lambdas_fine, scores_fine, 'tab:orange', lw=2.5, marker='s', ms=7,
            label='CV MSE (Fine)')
ax.axvline(melhor_fine, color='red', ls='--', lw=2, label=f'Melhor fine: $\\lambda={melhor_fine:.4f}$')
ax.set_xlabel('$\\lambda$ (Log-Scale)'); ax.set_ylabel('MSE (CV)')
ax.set_title(f'Passo 2: Fine Sweep\n(ao redor de $\\lambda={melhor_coarse:.4f}$)')
ax.legend(fontsize=9)

plt.suptitle('Estratégia Coarse-to-Fine com Log-Grid\nPor que log-scale? Cada salto = ordem de magnitude',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Avaliação final
modelo_otimo = Ridge(alpha=melhor_fine)
modelo_otimo.fit(X_tr_t, y_tr_t)
mse_final = mean_squared_error(y_te_t, modelo_otimo.predict(X_te_t))
print(f"λ ótimo (coarse): {melhor_coarse:.4f}")
print(f"λ ótimo (fine):   {melhor_fine:.4f}")
print(f"MSE no teste com λ={melhor_fine:.4f}: {mse_final:.4f}")

In [ ]:
# Diagnóstico: Under-Regularization vs. Over-Regularization
np.random.seed(5)
X_diag = np.sort(np.random.uniform(0, 1, 60)).reshape(-1, 1)
y_diag = np.sin(2*np.pi*X_diag.ravel()) + np.random.normal(0, 0.2, 60)

tamanhos_N = np.arange(10, 61, 5)
lambdas_diag = [1e-6, 1e-1, 10.0]  # under, ótimo, over
labels_diag  = ['$\\lambda=10^{-6}$\n(Under-Regularization\nHigh Variance)',
                '$\\lambda=0.1$\n(Boa Regularização)',
                '$\\lambda=10$\n(Over-Regularization\nHigh Bias)']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, lam, label in zip(axes, lambdas_diag, labels_diag):
    erros_tr, erros_val = [], []
    for N in tamanhos_N:
        X_n, y_n = X_diag[:N], y_diag[:N]
        modelo_d = make_pipeline(PolynomialFeatures(8), Ridge(alpha=lam))
        cv_s = cross_val_score(modelo_d, X_n, y_n, cv=min(5, N),
                               scoring='neg_mean_squared_error')
        modelo_d.fit(X_n, y_n)
        erros_tr.append(mean_squared_error(y_n, modelo_d.predict(X_n)))
        erros_val.append(-cv_s.mean())

    ax.plot(tamanhos_N, erros_tr,  'tab:green', lw=2, label='Erro Treino')
    ax.plot(tamanhos_N, erros_val, 'tab:red',   lw=2, label='Erro Validação')
    ax.fill_between(tamanhos_N, erros_tr, erros_val, alpha=0.15, color='tab:red')
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('N (amostras de treino)')
    ax.set_ylabel('MSE')
    ax.set_ylim(0, 1.0)
    ax.legend(fontsize=9)

plt.suptitle('Diagnóstico: Under vs. Over-Regularization via Learning Curves',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Resumo completo: comparação de todas as estratégias de regularização
print("=" * 70)
print("RESUMO: ESTRATÉGIAS DE REGULARIZATION")
print("=" * 70)

tabela = [
    ('Hard Reg.', '$\\theta \\in C$', 'Boundary geométrico', 'Proj. Grad. Desc.', 'SVM, norm constrain'),
    ('L2 / Ridge', '$\\lambda\\|\\theta\\|^2$', 'Círculo / bola', 'Fechada: $(X^TX+\\lambda I)^{-1}X^Ty$', 'Regressão, redes (Weight Decay)'),
    ('L1 / Lasso', '$\\lambda\\|\\theta\\|_1$', 'Diamante', 'ISTA, coord. descent', 'Feature Selection'),
    ('Elastic Net', '$\\lambda_1\\|\\theta\\|_1+\\lambda_2\\|\\theta\\|^2$', 'Híbrido', 'ADMM', 'Features correlacionadas'),
    ('Early Stop', 'Parar em época $t$', 'Trajetória curta', 'Monitor val. loss', 'Deep Learning'),
    ('Dropout', 'Mascarar $p$% neurônios', 'Ensemble implícito', 'Bernoulli mask', 'Redes neurais profundas'),
    ('Label Smooth', '$y_k \\leftarrow y_k(1-\\epsilon)+\\epsilon/K$', 'Suavização do target', 'Modificar labels', 'Classificação, Transformers'),
]

print(f"{'Técnica':<14} {'Penalidade':<32} {'Prior Implícito':<20} {'Quando usar'}")
print("-" * 70)
for nome, pen, geo, impl, quando in tabela:
    print(f"{nome:<14} {geo:<20} {quando}")

print()
print("CONEXÃO BAYESIANA:")
print("  Gaussian Prior (τ²)  →  L2 com λ = σ²/τ²")
print("  Laplace Prior  (b)   →  L1 com λ = σ²/b")
print()
print("LOG-GRID PARA HYPERPARAMETER TUNING:")
print("  Coarse: np.logspace(-5, 5, 20)")
print("  Fine:   np.logspace(log10(λ*)-1, log10(λ*)+1, 30)")
print("  Nunca use escala linear para λ!")

---

## Resumo Final — O que aprendemos

| Conceito | Ideia Central | Fórmula Chave |
|----------|---------------|---------------|
| **Hard Regularization** | Restrição estrita do espaço de parâmetros | $\theta \in C \subset \mathbb{R}^d$ |
| **Soft Regularization** | Penalidade adicionada à Loss Function | $\mathcal{L}(\theta) + \lambda\mathcal{R}(\theta)$ |
| **Output Regularization** | Penaliza propriedades da saída | $\mathcal{L}(\theta) + \mathcal{R}(y_i)$ |
| **L2 / Ridge** | Bola euclidiana — encolhimento suave | $\lambda\|\theta\|^2$ → Gaussian Prior |
| **L0 Sparsity** | Conta zeros — NP-hard | $\|\theta\|_0$ (inviável) |
| **L1 / Lasso** | Diamante — convexificação do L0 | $\lambda\|\theta\|_1$ → Laplace Prior |
| **Elastic Net** | Combina L1 + L2 | $\lambda_1\|\theta\|_1 + \lambda_2\|\theta\|^2$ |
| **Early Stopping** | Parar antes do overfitting | Equivalente a L2 para modelos lineares |
| **Dropout** | Zerar neurônios aleatoriamente | Ensemble implícito de $2^d$ redes |
| **Label Smoothing** | Suavizar os targets | $y_k^{\text{smooth}} = y_k(1-\epsilon) + \epsilon/K$ |
| **Bayesian Connection** | Regularização = Prior Bayesiano | MAP = Likelihood + Prior |
| **K-Fold CV** | Estimativa confiável do erro real | $\text{CV}_K = \frac{1}{K}\sum_k \text{MSE}_k$ |
| **Log-Grid Tuning** | Busca em ordens de magnitude | `np.logspace(-5, 5, 20)` |

---

## Metacognição — A Pergunta Certa sobre Log-Grid

> *"Por que usamos um Log-Grid ($10^{-4}, 10^{-3}, 10^{-2}$) em vez de escala linear ($0.1, 0.2, 0.3$)?"*

Na matemática do aprendizado de máquina, o impacto de forças de restrição opera em **ordens de magnitude**. Mudar $\lambda$ de $0.1$ para $0.2$ altera muito pouco a dinâmica do modelo. Mas saltar de $10^{-4}$ para $10^{-2}$ pode **transformar completamente** um modelo instável em uma solução altamente generalizável. O Log-Grid garante que você explore o espaço de forma uniforme onde realmente importa.

---

## Exercícios para Fixar

1. **L0 vs. L1:** crie um dataset com 100 features mas apenas 3 relevantes. Compare quantas features o Lasso consegue zerar vs. quantas o Ridge zera exatamente. Qual se aproxima mais da restrição L0 ideal?

2. **Bayesian Connection:** implemente manualmente o estimador MAP com Gaussian Prior e compare numericamente com Ridge (eles devem ser iguais para $\lambda = \sigma^2/\tau^2$).

3. **Early Stopping vs. Ridge:** usando a simulação da Seção 4a, verifique se o número de épocas do Early Stopping é compatível com $t \approx 1/(\eta\lambda)$ para algum $\lambda$ do Ridge.

4. **Stratified vs. K-Fold:** com um dataset desbalanceado (5% positivos), compare a variabilidade das estimativas do Stratified K-Fold vs. K-Fold padrão. O Stratified é mais estável?

5. **Coarse-to-Fine em Lasso:** aplique a estratégia Log-Grid coarse-to-fine ao Lasso. Como o número de coeficientes não-zero muda com $\lambda$ ao longo do grid?

---

### Referências neste repositório

- `L2_Bias_Variance_Model_Selection.ipynb` — Bias-Variance Trade-Off e onde a regularização se encaixa
- `visao_probabilistica.ipynb` — Perspectiva probabilística (MAP, likelihood, prior)
- `lectures/Statistisches_Lernen_2_bbm_v3.pdf` — Slides originais desta aula
- Hastie, Tibshirani & Friedman: *The Elements of Statistical Learning* — Capítulos 3, 7 e 12
- Bishop: *Pattern Recognition and Machine Learning* — Capítulo 3 (Bayesian Linear Regression)